In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind

In [ ]:
import constants as c
PAPER_PATH = c.PAPER_PATH

In [3]:
def code_choice(row):
    # Unannotated (NaN choice) rows are excluded from metrics: return NaN so
    # they can be dropped before computing PPV/FOR (committed table values were
    # computed on annotated rows only).
    if pd.isna(row):
        return np.nan
    row = str(row).lower()
    if 'not' in row: 
        return 0
    elif 'drivable' in row:
        return 0
    else: 
        return 1

In [4]:
def process_baseline(filepath, name, resample_to_balanced=False):
    df = pd.read_csv(filepath)
    df['gt'] = df['choice'].apply(code_choice)
    # Exclude unannotated (NaN choice) rows from the metric computation.
    df = df.dropna(subset=['gt'])
    
    # Handle different column names for predictions
    if 'nli_is_flooded' in df.columns:
        df['pred'] = df['nli_is_flooded'].astype(int)
    elif 'nli_label' in df.columns:
        # For Liu FloodVision, the positive label (flooded/actionable) is 'no, not passable'
        # All other labels ('uncertain', 'yes, passable') are considered negative
        df['pred'] = df['nli_label'].apply(lambda x: 1 if 'no, not passable' in str(x).lower() else 0)
    elif 'sentiment_1' in df.columns:
        # For inspection set, we have two questions (sentiment_1 and sentiment_2)
        if 'Basic' in name:
            df['pred'] = df['sentiment_2'].astype(int)
        else:
            df['pred'] = df['sentiment_1'].astype(int)
            
    if resample_to_balanced:
        # Resample to 250 images per predicted class (pred=1 and pred=0)
        df_pos = df[df['pred'] == 1].sample(n=250, random_state=42)
        df_neg = df[df['pred'] == 0].sample(n=250, random_state=42)
        df = pd.concat([df_pos, df_neg]).reset_index(drop=True)
    
    df['correct'] = df['pred'] == df['gt']
    
    print(f"--- {name} ---")
    print(f"GT Mean: {df['gt'].mean():.3f}")
    print(f"Correct Mean: {df['correct'].mean():.3f}")
    
    df_1 = df[df['pred'] == 1]
    df_0 = df[df['pred'] == 0]
    print(f"Counts: Pos={len(df_1)}, Neg={len(df_0)}")
    print(f"P(correct|pred=1): {df_1['correct'].mean():.3f}")
    print(f"P(correct|pred=0): {df_0['correct'].mean():.3f}")
    print()
    
    return df, df_1, df_0

In [5]:
lyu_basic, lyu_basic_1, lyu_basic_0 = process_baseline("../../data/revisions/prompt_baseline_annotations/lyu_basic_annotated.csv", "Lyu Basic")
lyu_adv, lyu_adv_1, lyu_adv_0 = process_baseline("../../data/revisions/prompt_baseline_annotations/lyu_advanced_annotated.csv", "Lyu Advanced")
yang_adv, yang_adv_1, yang_adv_0 = process_baseline("../../data/revisions/prompt_baseline_annotations/yang_advanced_annotated.csv", "Yang Advanced")
liu_fv, liu_fv_1, liu_fv_0 = process_baseline("../../data/revisions/prompt_baseline_annotations/liu_floodvision_annotated.csv", "Liu FloodVision")
insp_basic, insp_basic_1, insp_basic_0 = process_baseline("../../data/processed/inspection_set.csv", "Ours (Basic Flooding Query)", resample_to_balanced=True)
insp_1ft, insp_1ft_1, insp_1ft_0 = process_baseline("../../data/processed/inspection_set.csv", "Ours (>1ft flooding)", resample_to_balanced=False)

--- Lyu Basic ---
GT Mean: 0.000
Correct Mean: 0.500
Counts: Pos=250, Neg=250
P(correct|pred=1): 0.000
P(correct|pred=0): 1.000

--- Lyu Advanced ---
GT Mean: 0.012
Correct Mean: 0.512
Counts: Pos=250, Neg=250
P(correct|pred=1): 0.024
P(correct|pred=0): 1.000

--- Yang Advanced ---
GT Mean: 0.256
Correct Mean: 0.748
Counts: Pos=250, Neg=250
P(correct|pred=1): 0.504
P(correct|pred=0): 0.992

--- Liu FloodVision ---
GT Mean: 0.016
Correct Mean: 0.504
Counts: Pos=250, Neg=250
P(correct|pred=1): 0.020
P(correct|pred=0): 0.988

--- Ours (Basic Flooding Query) ---
GT Mean: 0.274
Correct Mean: 0.766
Counts: Pos=250, Neg=250
P(correct|pred=1): 0.540
P(correct|pred=0): 0.992

--- Ours (>1ft flooding) ---
GT Mean: 0.333
Correct Mean: 0.825
Counts: Pos=500, Neg=500
P(correct|pred=1): 0.658
P(correct|pred=0): 0.992



In [6]:
def compare_baselines(name1, df1_1, df1_0, name2, df2_1, df2_0):
    print(f"=== T-Test Comparison: {name1} vs {name2} ===\n")

    # Predicted Positives Comparison
    t_pos, p_pos = ttest_ind(df1_1['correct'], df2_1['correct'])
    print(f"Predicted Positives (pred=1):")
    print(f"  t = {t_pos:.4f}, p = {p_pos:.4e}")
    print(f"  {name1} p(correct) = {df1_1['correct'].mean():.3f}")
    print(f"  {name2} p(correct) = {df2_1['correct'].mean():.3f}")
    print()

    # Predicted Negatives Comparison
    t_neg, p_neg = ttest_ind(df1_0['correct'], df2_0['correct'])
    print(f"Predicted Negatives (pred=0):")
    print(f"  t = {t_neg:.4f}, p = {p_neg:.4e}")
    print(f"  {name1} p(correct) = {df1_0['correct'].mean():.3f}")
    print(f"  {name2} p(correct) = {df2_0['correct'].mean():.3f}")
    print("\n")


compare_baselines("Ours (>1ft flooding)", insp_1ft_1, insp_1ft_0, "Ours (Basic Flooding Query)", insp_basic_1, insp_basic_0)
compare_baselines("Ours (>1ft flooding)", insp_1ft_1, insp_1ft_0, "Liu FloodVision", liu_fv_1, liu_fv_0)
compare_baselines("Ours (>1ft flooding)", insp_1ft_1, insp_1ft_0, "Lyu Basic", lyu_basic_1, lyu_basic_0)
compare_baselines("Ours (>1ft flooding)", insp_1ft_1, insp_1ft_0, "Lyu Advanced", lyu_adv_1, lyu_adv_0)
compare_baselines("Ours (>1ft flooding)", insp_1ft_1, insp_1ft_0, "Yang Advanced", yang_adv_1, yang_adv_0)


=== T-Test Comparison: Ours (>1ft flooding) vs Ours (Basic Flooding Query) ===

Predicted Positives (pred=1):
  t = 3.1529, p = 1.6810e-03
  Ours (>1ft flooding) p(correct) = 0.658
  Ours (Basic Flooding Query) p(correct) = 0.540

Predicted Negatives (pred=0):
  t = 0.0000, p = 1.0000e+00
  Ours (>1ft flooding) p(correct) = 0.992
  Ours (Basic Flooding Query) p(correct) = 0.992


=== T-Test Comparison: Ours (>1ft flooding) vs Liu FloodVision ===

Predicted Positives (pred=1):
  t = 20.7888, p = 4.1076e-76
  Ours (>1ft flooding) p(correct) = 0.658
  Liu FloodVision p(correct) = 0.020

Predicted Negatives (pred=0):
  t = 0.5364, p = 5.9183e-01
  Ours (>1ft flooding) p(correct) = 0.992
  Liu FloodVision p(correct) = 0.988


=== T-Test Comparison: Ours (>1ft flooding) vs Lyu Basic ===

Predicted Positives (pred=1):
  t = 21.9023, p = 1.5317e-82
  Ours (>1ft flooding) p(correct) = 0.658
  Lyu Basic p(correct) = 0.000

Predicted Negatives (pred=0):
  t = -1.4180, p = 1.5660e-01
  Ours (>1ft 

/share/ju/matt/bayflood/.venv/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


In [7]:
def get_metrics(df, name):
    tp = len(df[(df['pred'] == 1) & (df['gt'] == 1)])
    fp = len(df[(df['pred'] == 1) & (df['gt'] == 0)])
    fn = len(df[(df['pred'] == 0) & (df['gt'] == 1)])
    tn = len(df[(df['pred'] == 0) & (df['gt'] == 0)])
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    csi = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    for_rate = fn / (fn + tn) if (fn + tn) > 0 else 0
    
    return {
        'Model': name,
        'PPV': precision,
        'FOR': for_rate
    }

results = [
    get_metrics(lyu_basic, "Lyu Basic"),
    get_metrics(lyu_adv, "Lyu Advanced"),
    get_metrics(liu_fv, "Liu FloodVision"),
    get_metrics(insp_basic, "Ours (Basic Flooding Query)"),
    get_metrics(insp_1ft, "Ours (>1ft flooding)"),
    get_metrics(yang_adv, "Yang Advanced (Ours)")
]

res_df = pd.DataFrame(results)
# Sort by Precision
res_df = res_df.sort_values('PPV', ascending=False).reset_index(drop=True)

metrics = ['PPV', 'FOR']
models = res_df['Model'].tolist()

max_precision_idx = res_df['PPV'].idxmax()
max_precision_model = res_df.loc[max_precision_idx, 'Model']

cols_str = 'l' + 'c' * len(metrics)
metrics_str = ' & '.join(metrics)

latex_table = f"""\\begin{{table}}[ht]
\\small
\\centering
\\caption{{Performance metrics across prompt-based baselines.}}
\\label{{tab:baseline_performance}}
\\begin{{tabular}}{{{cols_str}}}
\\toprule
Model & {metrics_str} \\\\
\\midrule
"""

for model in models:
    row_vals = []
    for metric in metrics:
        val = res_df[res_df['Model'] == model][metric].values[0]
        
        # Format the value
        if metric == 'FOR':
            if val == 0:
                formatted_val = "0.000"
            elif val < 0.001:
                exponent = int(np.floor(np.log10(val)))
                mantissa = val / (10**exponent)
                formatted_val = f"({mantissa:.2f} \\pm 0.00) \\cdot 10^{{{exponent}}}"
            else:
                formatted_val = f"{val:.3f}"
        else:
            formatted_val = f"{val:.3f}"
            
        # Bold the best model's row values
        if model == max_precision_model:
            formatted_val = f"\\textbf{{{formatted_val}}}"
            
        row_vals.append(formatted_val)
    
    # Bold the model name if it's the best one
    model_name = model
    if model == max_precision_model:
        model_name = f"\\textbf{{{model_name}}}"
        
    row_vals_str = ' & '.join(row_vals)
    latex_table += f"{model_name} & {row_vals_str} \\\\ \n"

latex_table += """\\bottomrule
\\end{tabular}
\\end{table}"""

latex_table += """\\bottomrule
\\end{tabular}
\\end{table}"""

with open(f"{PAPER_PATH}/tables/prompt_baselines.tex", "w") as f:
    f.write(latex_table)

print(latex_table)

\begin{table}[ht]
\small
\centering
\caption{Performance metrics across prompt-based baselines.}
\label{tab:baseline_performance}
\begin{tabular}{lcc}
\toprule
Model & PPV & FOR \\
\midrule
\textbf{Ours (>1ft flooding)} & \textbf{0.658} & \textbf{0.008} \\ 
Ours (Basic Flooding Query) & 0.540 & 0.008 \\ 
Yang Advanced (Ours) & 0.504 & 0.008 \\ 
Lyu Advanced & 0.024 & 0.000 \\ 
Liu FloodVision & 0.020 & 0.012 \\ 
Lyu Basic & 0.000 & 0.000 \\ 
\bottomrule
\end{tabular}
\end{table}\bottomrule
\end{tabular}
\end{table}
